![Cleaning data](Cleansing-Data.png)

## Six steps to clean data

Clean the **data structure**

1. **Scrape** the document (into a DataFrame)
2. Delete extra **rows**
3. Fix **column** structure

Clean the **data content**

4. Handle **missing** values
5. **Standardize** values
6. **Extract** unstructured data

1. Digitize the document. (format: PDF/Word/..., encoding: UTF-8/CP1252/...)
1. Delete extra rows (header, footer, total, subtotal, column number indicators, blank)
1. Fix column structure (missing headers, rename columns, mis-alignment)
1. Handle missing / duplicate values
  - De-duplicate rows (based on subset of data)
  - Identify (e.g. blank, NA, XX, 999, etc)
  - Ignore missing values
  - Remove rows with NA (subsetting) or columns with NA
  - Fill missing values:
    - Constant (e.g. zero, today, etc)
    - Column (e.g. created date  updated date)
    - Function (e.g. average of rows/columns)
    - External data
1. Standardize values
  - Convert text to numbers or dates
  - Remove extra characters (commas from numbers, prefixes, suffixes, spaces)
  - Correct mis-spellings
1. Extract unstructured data
  - Extract numbers with units (lbs & kgs, kph & mph)
  - Extract lists (e.g. multiple values in cells)
  - Extract dates (e.g. 25/10/16 or Oct 25, 2016)
  - Extract entities (e.g. names, places, phone numbers, PIN codes, things)
  - Extract name structure (e.g. title, first name, last name)

## Let's clean this analyst report

[Open file](Cleansing-Data-ABN-AMRO-report.pdf)

![ABN Amro Analyst Report](Cleansing-Data-ABN-Maro-report.png)

## 1. Scrape the document

Convert data in formats like PDF, HTML, Word, etc into a Python DataFrame

In [ ]:
import tabula
data1 = tabula.read_pdf('Cleansing-Data-ABN-AMRO-report.pdf', pages=3)
data1

## 2. Delete extra rows

Header, footer, total, subtotal, column number indicators, blank rows, etc.

In [ ]:
data2 = (data1
         .set_index('Unnamed: 0')
         .drop([pd.np.nan, 'Operating businesses', 'Group', 'Shares outs.', 'Per share'])
        )
data2

# 3. Fix column structure

Missing header row, column names split across rows / columns, mis-aligned columns, etc

In [ ]:
data3 = data2.copy()
data3.columns = ['RWAs', 'Capital allocated', 'Net profit 2004A', 'Net profit 2005A', 'Net profit 2006E', 'Net profit 2007E', 'As % 2006E', 'Peer group PE06E', 'Valuation', 'As %']
data3.index.name = 'Business'
data3

# 4. Handle missing values

- Identify missing values (blank, NA, XX, 999, ...)
- Plan a strategy to handle them
  - Do nothing
  - Remove rows / columns with missing values
  - Fill with a constant (e.g. zero, today, etc)
  - Fill from another column (e.g. defaults)
  - Fill using a function (e.g. average)
  - Fill with external data

In [ ]:
data4 = data3.replace('na', pd.np.nan)
data4

# 5. Standardize values

- Convert text to numbers or dates
- Remove extra characters (prefixes, suffixes, spaces)
- Correct mis-spellings

In [ ]:
data5 = (data4
        .replace(',', '', regex=True)                # Remove commas
        .replace('\((\d+)\)', r'-\1', regex=True)    # Convert (10) to -10
        .replace('%$', '', regex=True)               # Convert % to numbers
        .astype(float)                               # Convert to numbers
        )
data5.sum()

## 6. Extract unstructured data

- Extract numbers with units (lbs & kgs, kph & mph)
- Extract lists (e.g. multiple values in cells)
- Extract dates (e.g. 25/10/16 or Oct 25, 2016)
- Extract entities (e.g. names, places, phone numbers, PIN codes, things)
- Extract name structure (e.g. title, first name, last name)

This dataset has none of those. Let's pick another dataset.

# Product dosage composition

This is a [random dataset](http://ravenbhelpharma.com/composition.xlsx) I found yesterday.

![Product dosage composition](Cleansing-Data-Pharma.png)

## Let's clean the data

In [ ]:
import pandas as pd
pharma = pd.read_excel('Cleansing-Data-composition.xlsx', skiprows=[0, 1]).set_index('S. No.')
pharma.head()

1. **Scrape** the document: **Done**. `pd.read_excel`.
2. Delete extra **rows**: **Done**. `skiprows=[0, 1]`
3. Fix **column** structure: **None**. It's OK

as well as in data **content**

4. Handle **missing** values: **None**.
5. **Standardize** values: TODO
6. **Extract** unstructured data: TODO

## 5. Standardize values

The "Dosage Form" column is not standardized.

In [ ]:
pharma.sort_values('Dosage Form', ascending=True)

## 5. Standardize values with fingerprints

In [ ]:
from jellyfish import metaphone, soundex, nysiis, match_rating_codex
metaphone('tablot')

In [ ]:
pharma['Dosage Form'].apply(metaphone)

In [ ]:
pharma.groupby(pharma['Dosage Form'].apply(soundex))['Dosage Form'].unique().tolist()

## 5. Standardize values: custom fingerprints

You can create your own fuzzy matching algorithms too.

For example, let's listen to a few [Hindi songs from AR Rahman](http://www.s-anand.net/blog/a-r-rahman-hindi-songs/)

![Hindi Songs Quiz](Cleansing-Data-Hindi-Songs.png)

## 5. Standardize values: custom fingerprints

For this, I used a custom fingerprinting algorithm suited for Hindi movies and artistes.

In [ ]:
def fingerprint_hindi(s):    
        s = s + ' '
        s = s.upper()                               # Upper case
        s = re.sub(r'\s+', ' ', s)                  # Single space
        s = re.sub(r'W', 'V', s)                    # Vivaah or Wivaah?
        s = re.sub(r'KSH', 'X', s)                  # Dixit or Deekshit?
        s = re.sub(r'Z', 'J', s)                    # Jabardast or Zabardast?
        s = re.sub(r'PH', 'F', s)                   # Phirangi or Firangi?
        s = re.sub(r'([KGCJTDPBS])H', r'\1', s)     # Dheeraj or Diraj?
        s = re.sub(r'(.)\1', r'\1', s)              # Kaala or Kala?
        s = re.sub(r'Y', '', s)                     # Koyi or Koi?
        s = re.sub(r'([AEIOU])N ', r'\1', s)        # Hein or Hai?
        s = re.sub(r'[AEIOU]', '', s)               # Rehman or Rahman?
        s = re.sub(r' ', '', s)                     # Remove spaces
        return s

In [ ]:
assert fingerprint_hindi('Meenakshi Dixit') == fingerprint_hindi('minaxi  deekshith') == 'MNXDXT'
assert fingerprint_hindi('Ye Dil Hai Mushkil') == fingerprint_hindi('e dil hein muskil') == 'DLHMSKL'

In [ ]:
# Here's an example of the OpenRefine fingerprint algorithm 
# Source: https://gist.github.com/cjdd3b/0386f139bb953f046c6e
import re
import string
from unidecode import unidecode    # install this

PUNCTUATION = re.compile('[%s]' % re.escape(string.punctuation))

def preprocess(string):
    return PUNCTUATION.sub('', string.strip().lower())

def unique_preserving_order(seq):
    seen = set()
    seen_add = seen.add
    return [x for x in seq if not (x in seen or seen_add(x))]
        
def fingerprint(s):
    s = preprocess(s)
    return unidecode(' '.join(unique_preserving_order(sorted(s.split()))))

def ngram_fingerprint(s, n=1):
    s = preprocess(s)
    return unidecode(''.join(unique_preserving_order(sorted([s[i:i + n] for i in range(len(s) - n + 1)]))))

## 5. Standardize values: constituencies

Let's apply fingerprinting to the list of assembly constituency names across years.

In [ ]:
assembly = pd.read_csv('D:/site/gramener.com/viz/india-election-data/assembly-elections/assembly.csv')
assembly

## 5. Standardize values: constituencies

This corrects phonetic spellings mistakes like:

- BHADRACHALAM, BHADRACHELAM, BHADRAHCALAM
- TUGHLAKABAD, TUGHLAQABAD, TUGHLAQUABAD, TUGLAKABAD
- RISHIVANDIYAM, RISHIVADIYAM, RISHIVANDIAM, RISHIVANDIAM, RISHIVANDIYAM

... and non-phonetic ones like:

- OOTACAMUND, UDHAGAMANDALAM
- POTTAPIDARAM, OTTAPIDARAM
- HARASAPUR, NARSAPUR

... but we were still left with a huge gap.

## A brief detour: Delimitation

Constituencies are defined in rough proportion to population.

- 1952: Constituencies defined using 1951 census
- 1962: Constituencies defined using 1961 census
- 1972: Constituencies defined using 1971 census
- 1976: Stop delimitation until 2000. States promoting family planning don't lose
- 2002: Extend till 2026. But apply intra-state delimitation. Preserve # seats per state
- 2008: Intra-state delimination was actually implemented

For each of these years, we have potential re-definition of constituencies.

The biggest nightmare this caused me was **Machilipatnam**.

## A brief detour: Machilipatnam

![Machilipatnam market](https://upload.wikimedia.org/wikipedia/commons/thumb/7/70/Machilipatnam_koneru_center_3.JPG/250px-Machilipatnam_koneru_center_3.JPG)

Has 2 different names:

- Masulipatnam (or Machilipatnam): derived from the Greek trade of "muslin" (plain woven) cloth
- Bandar: derived from the Turkish word for "port"

In 2008, the Bandar constituency was *renamed* to Machilipatnam.

- Machilipatnam was NOT a new constituency
- Bandar was not removed from the list
- So history needs to be preserved

## 5. Standardize values: analysis

This is where the data itself came in handy.

![Assembly elections constituencies](Cleansing-Data-assembly-elections.png)

## 6. Extract values

- Extract numbers with units (lbs & kgs, kph & mph)
- Extract lists (e.g. multiple values in cells)
- Extract dates (e.g. 25/10/16 or Oct 25, 2016)
- Extract entities (e.g. names, places, phone numbers, PIN codes, things)
- Extract name structure (e.g. title, first name, last name)

## 6. Extract values: composition list

In [ ]:
pharma['Composition'].head()

In [ ]:
components = pharma['Composition'].str.split(r'\s*[,\+]\s*').apply(pd.Series).stack()
components.head(10)

## 6. Extract values: lists

In [ ]:
extracts = components.str.extract('(.*?)\s*(\d+)\s*(ml|mg|mcg|g|gm).*')
extracts['text'] = components
extracts.head()

## Don't overdo it

![Mess](https://imgs.xkcd.com/comics/mess.png)

# Let's automate as open source

We're creating an open source tool on **[Github: gramener/dataaudit](https://github.com/gramener/dataaudit)**.

(See the **dev** branch for now.)

We'll start with cleaning public health data at UP & Bihar.

Join in!